# RayGNN v0.1: Kaggle feasibility run

This notebook trains the fixed-White RayGNN directly from Stockfish `.binpack` data. The records preserve board state, side to move, castling, en-passant and halfmove clock. They do not preserve repetition history, so this run explicitly uses repetition count zero.

In [ ]:
from pathlib import Path
import datetime, json, os, random, shlex, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/lualum/nnue-pytorch.git'
BRANCH = 'experiment/raygnn-v0.1'
COMMIT = None  # Set a full SHA to pin a later rerun.
REPO = Path('/kaggle/working/nnue-pytorch')
RUNS = Path('/kaggle/working/raygnn-runs')

def run(args, cwd=None, env=None):
    args = [str(arg) for arg in args]
    print(shlex.join(args), flush=True)
    subprocess.run(args, cwd=cwd, env=env, check=True)

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO])
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
    if dirty:
        raise RuntimeError('Existing checkout has edits. Use a fresh Kaggle session.')
if COMMIT is not None:
    run(['git', 'checkout', '--detach', COMMIT], cwd=REPO)
print('Source:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
RUNS.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
print('Kaggle PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before starting real-data training.')
print('GPU 0:', torch.cuda.get_device_name(0))

PYTHON = Path(sys.executable)
constraint = Path('/tmp/raygnn-torch-constraint.txt')
constraint.write_text(f'torch=={torch.__version__}\n')
run([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
     '--upgrade-strategy', 'only-if-needed', '-c', constraint,
     'tyro>=0.9,<2', 'torchmetrics>=1.7,<2', 'tensorboard>=2,<3',
     'numba>=0.61', 'pytest>=8,<10', 'cmake>=3.20', 'zstandard>=0.23,<1',
     'chess>=1.11,<2'])

ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['TORCH_COMPILE_DISABLE'] = '1'
run(['cmake', '-S', 'data_loader/cpp', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release'], cwd=REPO, env=ENV)
run(['cmake', '--build', 'build', '--target', 'training_data_loader', '-j', '2'], cwd=REPO, env=ENV)
run([PYTHON, '-m', 'pytest', 'tests/test_raygnn.py', '-q'], cwd=REPO, env=ENV)

In [ ]:
import zstandard as zstd

MIN_REAL_BYTES = 10 * 1024 * 1024
MAX_EXTRACTED_BYTES = 38 * 2**30
DATA_DIR = Path('/kaggle/working/binpack-data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_NAME = 'test80-2024-06-jun-2tb7p.min-v2.v6.binpack'
DATA_URL = ('https://huggingface.co/datasets/linrock/test80-2024/resolve/main/'
            + DATA_NAME + '.zst?download=true')
ARCHIVE = DATA_DIR / (DATA_NAME + '.zst')
DOWNLOADED_BINPACK = DATA_DIR / DATA_NAME

attached = sorted((path for path in Path('/kaggle/input').rglob('*.binpack')
                   if path.stat().st_size >= MIN_REAL_BYTES),
                  key=lambda path: path.stat().st_size, reverse=True)
if attached:
    available = attached
    print('Using attached binpack:', *available, sep='\n  ')
else:
    if not DOWNLOADED_BINPACK.exists():
        if shutil.disk_usage(DATA_DIR).free < 12 * 2**30:
            raise RuntimeError('Insufficient disk. Attach a decompressed .binpack instead.')
        run(['wget', '-c', '--progress=dot:giga', DATA_URL, '-O', ARCHIVE])
        written = 0
        try:
            with ARCHIVE.open('rb') as src, DOWNLOADED_BINPACK.open('wb') as dst:
                with zstd.ZstdDecompressor().stream_reader(src) as reader:
                    while block := reader.read(16 * 1024 * 1024):
                        written += len(block)
                        if written > MAX_EXTRACTED_BYTES:
                            raise RuntimeError('Decompressed data exceeded the 38 GiB cap.')
                        dst.write(block)
        except Exception:
            DOWNLOADED_BINPACK.unlink(missing_ok=True)
            raise
        ARCHIVE.unlink(missing_ok=True)
    available = [DOWNLOADED_BINPACK]

TRAIN_FILES = [str(path) for path in available]
VALID_FILES = TRAIN_FILES.copy()  # Stability-only validation stream, not held out.
BATCH_SIZE = 256
EPOCH_SIZE = 131_072
MAX_EPOCHS = 80
VALIDATION_SIZE = 262_144
CHECK_VALID_EVERY = 5
MAX_TIME = '00:01:15:00'
SEED = 42

In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = RUNS / f'raygnn-binpack-t4-{stamp}'
RUN_DIR.mkdir(parents=True)

cmd = [str(PYTHON), 'train_raygnn.py', *TRAIN_FILES, '--binpack',
       '--validation-datasets', *VALID_FILES,
       '--batch-size', str(BATCH_SIZE), '--epoch-size', str(EPOCH_SIZE),
       '--max-epochs', str(MAX_EPOCHS), '--validation-size', str(VALIDATION_SIZE),
       '--check-val-every-n-epoch', str(CHECK_VALID_EVERY),
       '--accelerator', 'cuda', '--num-workers', '1', '--seed', str(SEED),
       '--max-time', MAX_TIME, '--save-every', '5', '--default-root-dir', str(RUN_DIR)]
manifest = {'source': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(),
            'command': cmd, 'torch_compile_disabled': True,
            'repetition_count': 0, 'validation_is_held_out': False,
            'data': [{'path': path, 'bytes': Path(path).stat().st_size} for path in TRAIN_FILES]}
(RUN_DIR / 'run_config.json').write_text(json.dumps(manifest, indent=2) + '\n')
(RUN_DIR / 'command.txt').write_text('TORCH_COMPILE_DISABLE=1 ' + shlex.join(cmd) + '\n')
started = time.perf_counter()
run(cmd, cwd=REPO, env=ENV)
elapsed = time.perf_counter() - started
(RUN_DIR / 'elapsed_seconds.txt').write_text(f'{elapsed:.3f}\n')
print(f'Finished in {elapsed / 60:.1f} minutes: {RUN_DIR}')

In [ ]:
import pandas as pd
from IPython.display import display, FileLink

metrics_path = RUN_DIR / 'metrics.csv'
if not metrics_path.exists():
    raise RuntimeError('No metrics.csv found. Inspect the training cell output.')
metrics = pd.read_csv(metrics_path)
display(metrics)
checkpoints = sorted(RUN_DIR.glob('*.ckpt'))
print('Checkpoints:', *checkpoints, sep='\n  ')
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
os.chdir('/kaggle/working')
display(FileLink(str(Path(archive).relative_to('/kaggle/working'))))